# Chapter 1: Understanding Functional Programming

In [62]:
from collections.abc import Callable, Generator, Iterator, Sequence

import ipytest
import pytest

ipytest.autoconfig()

## Identifying a paradigm

* The concept of **state** acting as the key differentiator between two paradigms—functional programming and imperative programming.
* **Imperative Programming:**
    * **State & Mutability:** State is defined by variable values across namespaces; statements act as explicit commands that *alter state* by adding, changing, or removing variables.
    * **Role of Statements:** Focuses on the assignment statement to *advance state*. Control flow statements (`if`, `for`, `while`, `try`) guard or repeat state changes, while statements like `def`, `class`, and `import` alter context.
    * **Program Verification:** Proving that sequential state changes reach the desired outcome is challenging; it often requires working backward from a defined final state to derive necessary preconditions.
* **Functional Programming:**
    * **Stateless Evaluation:** Replaces mutable state with function evaluations, creating new objects from existing ones rather than modifying variables in place.
    * **Modular Composition:** Composes small, easy-to-understand lower-level functions into higher-level abstractions that are simpler to visualize than complex imperative statement sequences.
    * **Mathematical Alignment & Reliability:** Parallels mathematical formalisms, enabling algebraic reasoning to handle edge cases, simplify algorithm design, and streamline unit testing.
    * **Code Characteristics:** Produces succinct, expressive, and efficient code through deliberate design, which is often easier to achieve than in imperative/procedural programming.

## Subdividing the procedural paradigm

* We can subdivide imperative languages into a number of discrete categories.
    * Object-oriented programming is a subset of imperative programming.
* To sum a range of numbers that share a common property:
    * Strictly procedural: See `test_sum_range_procedural()`
    * Object-oriented: See `test_sum_range_object_oriented()`
    * Object-oriented (pedantic): See `test_sum_range_object_oriented_pedantic()`
* All 3 of these examples rely on variables to explicitly show the state of the program.
    * They rely on the assignment statements to change the values of the variables and advance the computation toward completion.

In [63]:
%%ipytest


def test_sum_range_procedural() -> None:
    """Sum a range of numbers - strictly procedural.

    The program's state is defined by the values of the variables `s` and `n`.
    The variable `n` takes on values such that `1 ≤ n < 10`. As the loop
    involves an ordered exploration of values of `n`, we can prove that it
    will terminate when `n == 10`.
    """
    total = 0
    for num in range(1, 10):
        if num % 3 == 0 or num % 5 == 0:
            total += num

    assert total == 23


def test_sum_range_object_oriented() -> None:
    """Sum a range of numbers - object-oriented.

    Accumulates a stateful collection object, `m`, as it proceeds. The state of the
    computation is defined by the values of the variables `m` and `n`.
    """
    m: list[int] = []
    for n in range(1, 10):
        if n % 3 == 0 or n % 5 == 0:
            m.append(n)

    total = sum(m)
    assert total == 23


def test_sum_range_object_oriented_pedantic() -> None:
    """Sum a range of numbers - object-oriented (pedantic).

    We could fully embrace the object model, by defining a subclass of the `list` class.
    """

    class SummableList(list[int]):
        def sum(self) -> int:
            total = 0
            for v in self:
                total += v
            return total

    m = SummableList()
    for n in range(1, 10):
        if n % 3 == 0 or n % 5 == 0:
            m.append(n)

    total = m.sum()
    assert total == 23

...                                                                                          [100%]
3 passed in 0.01s


## Using the functional paradigm

* The sum of the multiples of 3 and 5 can be defined in 2 parts:
    * The sum of a sequence of numbers.
    * A sequence of values that pass a simple test condition.
* The sum of a sequence has a simple, **recursive definition**.

In [64]:
%%ipytest


def sum_recursive(seq: Sequence[int]) -> int:
    if len(seq) == 0:
        return 0

    return seq[0] + sum_recursive(seq[1:])


def test_sum_recursive() -> None:
    total = sum_recursive([1, 2, 3, 4])
    assert total == 10

.                                                                                            [100%]
1 passed in 0.00s


* We've defined the **sum of a sequence** in 2 cases:
    * the **base case** states that the sum of a zero length sequence is 0,
    * the **recursive case** states that the sum of a sequence is the first value plus the sum of the rest of the sequence.
* Since the recursive definition depends on a shorter sequence, we can be sure that it will (eventually) devolve to the base case.
* A **sequence of values** can have a simple, recursive definition.

In [65]:
def until(
    upper_bound: int,
    filter_func: Callable[[int], bool],
    value: int,
) -> list[int]:
    if value == upper_bound:
        return []
    if filter_func(value):
        return [value] + until(upper_bound, filter_func, value + 1)

    return until(upper_bound, filter_func, value + 1)


* There are 3 cases in this function:
    * Base case: If `value` reaches `upper_bound`, the resulting list must be empty.
    * If `value` is passed by `filter_func`, we'll create a very small list containing one element, and append the remaining values of the `until` function to this list.
    * If `value` is rejected by `filter_func`, this value is ignored and the result is simply defined by the remaining values of the `until` function.
* `value` will increase from an initial value until it reaches `upper_bound`, assuring us that we'll reach the base case soon.
* To generate the multiples of 3 and 5:

In [66]:
%%ipytest

mult_3_5 = lambda n: n % 3 == 0 or n % 5 == 0


def test_until() -> None:
    assert until(10, mult_3_5, 1) == [3, 5, 6, 9]


.                                                                                            [100%]
1 passed in 0.00s


* We can use our recursive `sum_recursive()` function to compute the sum of this sequence of values.
* Functions such as `sum_recursive()` and `until()` are defined as simple recursive functions.
    * The values are computed without resorting to using intermediate variables to store the state.

In [67]:
%%ipytest


def test_sum_until() -> None:
    total = sum_recursive(until(10, mult_3_5, 1))
    assert total == 23


.                                                                                            [100%]
1 passed in 0.00s


## Using a functional hybrid

* Our hybrid functional version might look like the following:

In [68]:
%%ipytest


def test_hybrid() -> None:
    total = sum(n for n in range(1, 10) if n % 3 == 0 or n % 5 == 0)

    assert total == 23

.                                                                                            [100%]
1 passed in 0.00s


* The `if` clause of the expression can be extracted into a separate function, allowing us to easily repurpose this for other rules.
* The variable `n` in this example isn't directly comparable to the variable `n` in the first two imperative examples.
    * A `for` statement (outside a generator expression) creates a proper variable in the local namespace.
* The variable `n` doesn't exist outside the binding in the generator expression.
    * It is is bound to each value, more as a way of expressing the contents of the set.
    * It doesn't define the state of the computation.

## Looking at object creation

* In some cases, it might help to look at intermediate objects as a history of the computation.
    * The history of a computation is not fixed.
* When functions are *commutative* or *associative*, then changes to the order of evaluation might lead to different objects being created.
    * Cummutative: `a + b = b + a` (order does not matter)
    * Associative: `(a + b) + c = a + (b + c)` (grouping does not matter)
* This might have performance improvements with no changes to the correctness of the results.
* However:
    * **Theoretical Models vs. Runtime Realities:** High-level time complexity models assume strict immutable copies (predicting $O(N^2)$ performance for repeated left-to-right string concatenation). CPython bypasses this with a C-level optimization: if an object's reference count is 1, it resizes the memory buffer in-place, executing in amortized $O(N)$ time.
    * **Sensitivity of Micro-Optimizations:** Runtime optimizations depend heavily on precise execution conditions. Simply keeping an extra reference to a variable breaks the refcount check, turning an $O(N)$ loop back into an $O(N^2)$ bottleneck.
    * **Always Measure First:** Intuition about evaluation order or theoretical performance can be invalidated by hidden runtime or compiler optimizations. Empirical benchmarking on the target environment is essential before attempting manual code restructurings.
* What's important for functional design is the idea that the `+` operator (or `add()` function) can be used in any order to produce the same results.

## A classic example of functional programming

* Example: the Newton-Raphson algorithm for locating the roots of a function, illustrating how the **iterative square root approximation** function works.
* Terminology for $\sqrt[i]{n}$:
    * $\sqrt{\phantom{x}}$: radical sign
    * $n$: radicand
    * $i$: index (the root being taken)
* For $a = \sqrt{n}$, the process relies on updating an initial estimate $a_i$ to get a better approximation $a_{i+1}$ using the formula:

$$a_{i+1} = \frac{a_i + n / a_i}{2}$$

* In Python, this update step can be defined as:

In [69]:
def next_(n: int, ai: float) -> float:
    return (ai + n / ai) / 2

* *Note: The function is named `next_()` with a trailing underscore to avoid shadowing Python's built-in `next()` function.*
* Each iteration refines the estimate by taking the average of $a_i$ and $n / a_i$.
* As the sequence converges, $a$ approaches a value where $a = n / a$, which implies $a^2 = n$, or $a = \sqrt{n}$.
* Here's how the function looks:

In [70]:
%%ipytest


def test_newton_raphson() -> None:
    n = 4

    a0 = 1.0
    a1 = next_(n, a0)
    a2 = next_(n, a1)
    a3 = next_(n, a2)

    assert (a0, a1, a2, a3) == pytest.approx(
        (1.0, 2.5, 2.05, 2.0006),
        abs=1e-5,
    )

    f = lambda ai: next_(n, ai)
    assert (a0, f(a0), f(f(a0)), f(f(f(a0)))) == pytest.approx(
        (1.0, 2.5, 2.05, 2.0006),
        abs=1e-5,
    )


.                                                                                            [100%]
1 passed in 0.00s


* We can write a function, which will (in principle) generate an infinite sequence of $a_i$ values converging on the proper square root:

In [71]:
def repeat(f: Callable[[float], float], a: float) -> Generator[float]:
    yield a
    for v in repeat(f, f(a)):
        yield v

* Step-by-step example execution of the `repeat()` function using:
    * `n = 4`
    * initial seed `a = 1.0`
    * `f = lambda ai: next_(4, ai)`
* **Step 1: The initial call — `repeat(f, 1.0)`**
    * `yield a`: The function immediately yields the first value, `1.0`, outward to the caller.
    * The function moves to the loop: `for v in repeat(f, f(a)):`. It calculates `f(1.0)`, which equals `2.5`.
    * This triggers the recursive call: `repeat(f, 2.5)`.
* **Step 2: First recursive call — `repeat(f, 2.5)`**
    * `yield a`: Inside this inner scope, `yield a` yields `2.5`.
    * Where `yield v` takes place:
        * The yielded `2.5` is caught by the `for v in ...` loop waiting in the *outer call* (`repeat(f, 1.0)`).
        * Because of this loop, `yield v` in the outer call executes, taking that `2.5` and passing it outward to the final consumer.
    * Meanwhile, the current (inner) call moves to its own loop, evaluating `f(2.5)`, which equals `2.05`.
    * This triggers the next recursive call: `repeat(f, 2.05)`.
* **Step 3: Second recursive call — `repeat(f, 2.05)`**
    * `yield a`: Inside this deeper scope, `yield a` yields `2.05`.
    * Where `yield v` takes place:
        * The yielded `2.05` is caught by the `for v in ...` loop of the *middle* call (`repeat(f, 2.5)`).
        * The middle call's `yield v` executes, passing it up to the *outer* call's loop.
        * The outer call's `yield v` then executes, finally emitting `2.05` to the external consumer.
    * The function continues evaluating `f(2.05)` to get `2.0006`, repeating the cascading `yield v` chain for the next iteration.
* `yield v`'s role:
    * Every time a deeper recursive layer yields its initial `a`, the `for v in ...` loop in the layer above catches it as `v` and instantly triggers its own `yield v` statement.
    * This creates a pipeline where values generated deep inside recursive loops are passed back up the chain and streamed outward one by one.
* It's essential to stop generating values when two values are so close to each other that either one is useful as the square root we're looking for.
* The common symbol for the value, which is close enough, is epsilon, ε -- the largest error we will tolerate.

In [72]:
%%ipytest


def within(epsilon: float, iterable: Iterator[float]) -> float:
    def head_tail(epsilon: float, a: float, iterable: Iterator[float]) -> float:
        b = next(iterable)
        if abs(a - b) <= epsilon:
            return b
        return head_tail(epsilon, b, iterable)

    return head_tail(epsilon, next(iterable), iterable)


def test_within() -> None:
    sqrt_4_approx_values = (1.0, 2.5, 2.05, 2.0006)
    assert within(0.5, iter(sqrt_4_approx_values)) == 2.05
    assert within(0.05, iter(sqrt_4_approx_values)) == 2.0006

.                                                                                            [100%]
1 passed in 0.00s


* We can use the three functions `next_()`, `repeat()`, and `within()` to create a square root function.

In [73]:
%%ipytest


def sqrt(a0: float, epsilon: float, n: int) -> float:
    return within(epsilon, repeat(lambda x: next_(n, x), a0))


def test_sqrt() -> None:
    assert sqrt(1.0, 0.5, 4) == 2.05
    assert sqrt(1.0, 0.05, 4) == pytest.approx(2.0006, abs=1e-5)
    assert sqrt(1.0, 0.005, 4) == pytest.approx(2.00000009, abs=1e-8)


.                                                                                            [100%]
1 passed in 0.00s
